# 02 · Educación

**Fuente:** INE, Encuesta Continua de Hogares 2024 (microdatos), tomada de la tabla `ech_2024_bronze` que crea `01_pobreza`. Todo está ponderado con `W_ANO`.

| Variable | Qué es |
|---|---|
| `e49` | Asistencia a la educación (3 = asiste actualmente) |
| `e201_1c` / `e201_1d` | Terminó bachillerato en liceo / en UTU (1 = sí) |
| `e51_8` a `e51_11` | Años aprobados en terciaria (magisterio, universidad, terciario, posgrado) |
| `POBPCOAC` | Condición de actividad (2 = ocupado, 3 a 5 = desocupado) |

**Indicadores:** egreso de educación media superior por generación y sexo; jóvenes de 15 a 24 que no estudian ni trabajan; desempleo de 18 a 29 según si terminaron el liceo.

## 1. Silver

"Terminó la media superior" = bachillerato completo en liceo o UTU, o estudios terciarios.

In [0]:
from pyspark.sql import functions as F

NOMBRES = {1: "Montevideo", 2: "Artigas", 3: "Canelones", 4: "Cerro Largo", 5: "Colonia", 6: "Durazno",
           7: "Flores", 8: "Florida", 9: "Lavalleja", 10: "Maldonado", 11: "Paysandú", 12: "Río Negro",
           13: "Rivera", 14: "Rocha", 15: "Salto", 16: "San José", 17: "Soriano", 18: "Tacuarembó", 19: "Treinta y Tres"}
mapa = F.create_map([F.lit(x) for kv in NOMBRES.items() for x in kv])

termino_media = ((F.col("e201_1c") == 1) | (F.col("e201_1d") == 1) | (F.col("e51_8") > 0) |
                 (F.col("e51_9") > 0) | (F.col("e51_10") > 0) | (F.col("e51_11") > 0))

educ = (spark.table("workspace.pobreza.ech_2024_bronze").select(
        F.col("e27").cast("int").alias("edad"),
        F.when(F.col("e26") == 1, "Hombres").otherwise("Mujeres").alias("sexo"),
        F.col("dpto").cast("int").alias("dpto"),
        mapa[F.col("dpto")].alias("departamento"),
        termino_media.alias("termino_media"),
        (F.col("e49") == 3).alias("estudia"),
        (F.col("POBPCOAC") == 2).alias("ocupado"),
        F.col("POBPCOAC").between(3, 5).alias("desocupado"),
        F.col("W_ANO").alias("peso")))

educ.write.mode("overwrite").option("overwriteSchema", True).saveAsTable("workspace.pobreza.educacion_silver")
display(educ.limit(5))

## 2. Validación contra cifras oficiales

Si las definiciones están bien, estos números tienen que parecerse a los publicados: egreso 21-23 ≈ 51,6% (INEEd 2023) y desempleo juvenil ≈ 25-27% (INE / OIT).

In [0]:
e = spark.table("workspace.pobreza.educacion_silver")
pct = lambda cond: F.round(F.sum(F.when(cond, F.col("peso")).otherwise(0)) / F.sum("peso") * 100, 1)

print("Egreso media 21-23 (INEEd 2023: 51,6%):",
      e.filter(F.col("edad").between(21, 23)).agg(pct(F.col("termino_media"))).first()[0])
print("Desempleo 14-24 (INE/OIT: 25-27%):",
      e.filter(F.col("edad").between(14, 24) & (F.col("ocupado") | F.col("desocupado"))).agg(pct(F.col("desocupado"))).first()[0])
print("Ni estudian ni trabajan 15-24:",
      e.filter(F.col("edad").between(15, 24)).agg(pct(~F.col("estudia") & ~F.col("ocupado"))).first()[0])

## 3. Gold + export

In [0]:
import os

def carpeta_data():
    ruta = os.path.dirname(dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get())
    if not ruta.startswith("/Workspace"):
        ruta = "/Workspace" + ruta
    if os.path.basename(ruta) == "notebooks":   # si estoy en la carpeta notebooks...
        ruta = os.path.dirname(ruta)            # ...subo un nivel, a la raíz del repo
    os.makedirs(f"{ruta}/data", exist_ok=True)
    return f"{ruta}/data"

def guardar_gold(df, nombre):
    df.write.mode("overwrite").option("overwriteSchema", True).saveAsTable(f"workspace.pobreza.{nombre}_gold")
    df.toPandas().to_json(f"{carpeta_data()}/{nombre}.json", orient="records", force_ascii=False, indent=1)
    print("OK:", nombre)

In [0]:
e = spark.table("workspace.pobreza.educacion_silver")
p = lambda cond: F.round(F.sum(F.when(cond, F.col("peso")).otherwise(0)) / F.sum("peso") * 100, 1)
n = F.count("*").alias("n_muestra")

# 1) Egreso de media superior por generación y sexo
gen = (F.when(F.col("edad").between(21, 23), "21-23").when(F.col("edad").between(25, 29), "25-29")
        .when(F.col("edad").between(30, 39), "30-39").when(F.col("edad").between(40, 49), "40-49")
        .when(F.col("edad").between(50, 64), "50-64").when(F.col("edad") >= 65, "65+"))
egreso = (e.withColumn("generacion", gen).filter(F.col("generacion").isNotNull())
          .groupBy("generacion").agg(p(F.col("termino_media")).alias("pct_total"), n))
egreso_sexo = (e.withColumn("generacion", gen).filter(F.col("generacion").isNotNull())
               .groupBy("generacion", "sexo").agg(p(F.col("termino_media")).alias("pct")))
egreso = egreso.join(egreso_sexo.groupBy("generacion").pivot("sexo").agg(F.first("pct")), "generacion").orderBy("generacion")

# 2) Ni estudian ni trabajan, 15-24
j = e.filter(F.col("edad").between(15, 24))
nini = ~F.col("estudia") & ~F.col("ocupado")
nini_sexo = j.groupBy("sexo").agg(p(nini).alias("pct_nini"), n).orderBy("sexo")
nini_dpto = j.groupBy("dpto", "departamento").agg(p(nini).alias("pct_nini"), n).orderBy("dpto")

# 3) Desempleo de 18-29 según si terminó el liceo
act = e.filter(F.col("edad").between(18, 29) & (F.col("ocupado") | F.col("desocupado")))
desempleo_educ = (act.withColumn("educacion", F.when(F.col("termino_media"), "Terminó el liceo").otherwise("No terminó el liceo"))
                  .groupBy("educacion").agg(p(F.col("desocupado")).alias("pct_desempleo"), n))

guardar_gold(egreso, "educacion_egreso_generacion")
guardar_gold(nini_sexo, "educacion_nini_sexo")
guardar_gold(nini_dpto, "educacion_nini_dpto")
guardar_gold(desempleo_educ, "educacion_desempleo")

display(egreso); display(nini_sexo); display(desempleo_educ)

In [ ]:
# 4) Actividad de los jóvenes de 18 a 29 según si terminaron el liceo
#    El desempleo solo mira a los que buscan trabajo. Acá se reparte a TODOS los jóvenes en tres grupos:
#    ocupados, desocupados (buscan y no consiguen) e inactivos (no trabajan ni buscan: estudian, cuidan, etc.).
jov = e.filter(F.col("edad").between(18, 29)).withColumn(
    "educacion", F.when(F.col("termino_media"), "Terminó el liceo").otherwise("No terminó el liceo"))
inactivo = ~F.col("ocupado") & ~F.col("desocupado")
actividad = (jov.groupBy("educacion")
             .agg(p(F.col("ocupado")).alias("pct_ocupados"),
                  p(F.col("desocupado")).alias("pct_desocupados"),
                  p(inactivo).alias("pct_inactivos"),
                  p(inactivo & ~F.col("estudia")).alias("pct_inactivos_no_estudian"),
                  n)
             .orderBy("educacion"))
display(actividad)
guardar_gold(actividad, "educacion_actividad")

## Hallazgos (2024)

- Solo el **53,2%** de los jóvenes de 21 a 23 años terminó la educación media superior.
- Cada generación termina más que la anterior: **22,8%** entre los mayores de 65. Las mujeres superan a los hombres en todas las generaciones (10 puntos entre los de 21 a 23).
- **13,2%** de los hombres y **17,0%** de las mujeres de 15 a 24 no estudian ni trabajan.
- Desempleo de 18 a 29: **22,8%** sin liceo completo y **17,6%** con liceo completo.